# Treinamento: atualização do Lattes com uma skill

Este laboratório ensina a preparar e conferir uma atualização com a skill `lattes-atualizar`. Execute as células na ordem. O modo padrão usa dados fictícios e funciona sem rede. Nenhuma célula faz login, preenche o Lattes ou altera XLSX.

**Objetivos:** entender as responsabilidades do agente, ler o inventário, distinguir os estados, identificar pendências, preparar uma proposta e conferir uma execução simulada.

**Pré-requisito:** kernel Python 3.10 ou posterior recomendado. As células usam apenas biblioteca padrão. Abra o notebook na raiz deste projeto. Instalação do editor: consulte [README.md](README.md).

**Entrega do exercício:** um relatório JSON em `.work/treinamento/` e um prompt para levar ao harness. O relatório distingue inventário real opcional de resultados simulados.

## 1. Configurar o laboratório

Mantenha `USAR_PLANILHA_REAL=False` na primeira execução. Ajuste `RAIZ_PROJETO` se o notebook foi aberto a partir de outra pasta. O diretório informado deve conter a pasta `lattes-atualizar`.

Para a segunda rodada, você poderá apontar para um XLSX real. A leitura é local e não confirma nada no Lattes. Nunca cole credenciais em uma célula.

**Resultado esperado:** caminho do projeto e identificação do modo fictício.

In [ ]:
from pathlib import Path
import copy
import hashlib
import importlib.util
import json
import re
import unicodedata
import uuid
from collections import Counter
from datetime import datetime, timezone

RAIZ_PROJETO = Path.cwd().resolve()  # Ajuste se necessário.
USAR_PLANILHA_REAL = False
CAMINHO_PLANILHA = Path("dados") / "Atualizacao Lattes.xlsx"
# Exemplo Windows: Path(r"D:\pasta\Atualizacao Lattes.xlsx")

assert (RAIZ_PROJETO / "lattes-atualizar" / "SKILL.md").is_file(), (
    "Ajuste RAIZ_PROJETO para a pasta que contém lattes-atualizar."
)
print("Projeto:", RAIZ_PROJETO)
print("Modo:", "leitura real, sem gravação" if USAR_PLANILHA_REAL else "dados fictícios")

## 2. Entender o contrato antes de processar

Leia o contrato abaixo. `Solicitado` não é autorização automática para editar o site. `Atualizado` é histórico, não prova de conferência nesta sessão. Vazio não é falso.

**Exercício:** explique por que uma linha com atualização marcada e solicitação vazia não deve ser apagada ou recadastrada automaticamente.

In [ ]:
contrato = RAIZ_PROJETO / "lattes-atualizar" / "references" / "planilha.md"
print(contrato.read_text(encoding="utf-8"))

## 3. Preparar dados fictícios

A estrutura abaixo imita o JSON do leitor XLSX: abas, coordenadas e valores. As pessoas e atividades são fictícias. As URLs em `example.org` são ilustrativas e não serão acessadas.

Há cinco atividades: uma pendência solicitada, uma declarada atualizada, um histórico sem solicitação, uma triagem e um estado inválido. Há também uma linha só com ano e uma linha vazia.

**Resultado esperado:** fixture criada, sem arquivos de planilha gerados.

In [ ]:
cabecalhos = ["Ano", "Tipo", "Subtipo", "SubSubtipo", "Local", "Aluno",
              "Obs", "Link", "Solicitado", "Atualizado", "Qualis"]
linhas_ficticias = [
    [2026, "Curso", "Curso demonstrativo", None, "Instituição Exemplo", None,
     "Participação como aluno, 8 horas", "https://example.org/certificado", True, False, None],
    [2026, "Artigo", "Periódico Exemplo", None, None, None,
     "Artigo demonstrativo", None, True, True, None],
    [2026, "Defesa", "Final", "Mestrado", "Instituição Exemplo", "Pessoa Fictícia A",
     "Papel do titular ainda não conferido", None, None, True, None],
    [2026, None, None, None, None, "Registro de software a esclarecer",
     None, None, None, None, None],
    [2026, "Orientação", None, None, "Instituição Exemplo", "Pessoa Fictícia B",
     None, None, "talvez", False, None],
    [2026] + [None] * 10,
    [None] * 11,
]

def linha_json(numero, valores):
    return {"row": numero, "cells": {
        f"{chr(65 + coluna)}{numero}": {"value": valor}
        for coluna, valor in enumerate(valores) if valor is not None
    }}

inventario_ficticio = {
    "sha256": None,  # Não existe XLSX de origem nesta simulação.
    "sheets": [{
        "name": "Treino",
        "rows": [linha_json(1, cabecalhos)] + [
            linha_json(i, valores) for i, valores in enumerate(linhas_ficticias, 2)
        ],
        "hyperlinks": []
    }]
}
print("Fixture pronta: cinco atividades, uma linha só com ano e uma vazia.")

## 4. Ler o XLSX real, apenas se você optar

Com o modo fictício, esta célula copia a fixture. Com o modo real, carrega o leitor do pacote, extrai os dados e compara o hash antes e depois para verificar a preservação do arquivo.

O leitor não recalcula fórmulas nem converte números seriais em datas. Se existirem fórmulas, o cache pode estar desatualizado: faça a conferência com ferramenta adequada antes de usar o resultado operacionalmente.

**Resultado esperado no modo padrão:** aba `Treino`. Não imprima dados pessoais desnecessários ao usar o modo real.

In [ ]:
if USAR_PLANILHA_REAL:
    caminho_real = CAMINHO_PLANILHA.expanduser().resolve()
    assert caminho_real.is_file(), f"Arquivo não encontrado: {caminho_real}"
    script = RAIZ_PROJETO / "lattes-atualizar" / "scripts" / "inspect_xlsx.py"
    spec = importlib.util.spec_from_file_location("leitor_lattes", script)
    leitor = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(leitor)
    hash_antes = hashlib.sha256(caminho_real.read_bytes()).hexdigest()
    inventario = leitor.inspect(caminho_real)
    assert hash_antes == hashlib.sha256(caminho_real.read_bytes()).hexdigest()
    origem = str(caminho_real)
else:
    inventario = copy.deepcopy(inventario_ficticio)
    origem = "fixture fictícia em memória"

print("Abas:", [aba["name"] for aba in inventario["sheets"]])
print("Origem preservada. Nenhuma atualização no Lattes foi executada.")

## 5. Interpretar cabeçalhos e linhas

O código procura `Ano`, `Solicitado` e `Atualizado` pelo nome, sem fixar letras de coluna. Ele separa linhas totalmente vazias e linhas sem descrição de atividade.

Esta é uma triagem didática: uma linha com apenas link ou observação ainda pode ser uma atividade incompleta e exige análise humana. Fórmulas são sinalizadas. Abas sem o contrato esperado causam erro explícito, em vez de serem descartadas silenciosamente.

**Resultado esperado na fixture:** cinco atividades e uma linha parcial.

In [ ]:
def normalizar_texto(valor):
    texto = unicodedata.normalize("NFKD", str(valor or ""))
    return " ".join("".join(c for c in texto if not unicodedata.combining(c)).casefold().split())

def letra(coordenada):
    return re.sub(r"\d+", "", coordenada)

def converter_aba(aba):
    obrigatorios = {"ano", "solicitado", "atualizado"}
    header = None
    for linha in aba["rows"]:
        nomes = {normalizar_texto(c.get("value")) for c in linha["cells"].values()}
        if obrigatorios <= nomes:
            header = linha
            break
    if header is None:
        raise ValueError(f"Aba {aba['name']}: cabeçalhos esperados não encontrados.")
    mapa = {letra(ref): str(c["value"]).strip()
            for ref, c in header["cells"].items() if c.get("value") is not None}
    normalizados = [normalizar_texto(nome) for nome in mapa.values()]
    if len(normalizados) != len(set(normalizados)):
        raise ValueError("Cabeçalhos duplicados: revisar antes de processar.")
    atividades, parciais = [], []
    campos_descritivos = {"tipo", "subtipo", "subsubtipo", "local", "aluno", "obs", "link"}
    for linha in aba["rows"]:
        if linha["row"] <= header["row"]:
            continue
        dados = {mapa[letra(ref)]: c.get("value")
                 for ref, c in linha["cells"].items() if letra(ref) in mapa}
        if not any(v is not None and v != "" for v in dados.values()):
            continue
        item = {"aba": aba["name"], "linha": linha["row"], "dados": dados,
                "formulas": [ref for ref, c in linha["cells"].items() if "formula" in c]}
        tem_descricao = any(normalizar_texto(k) in campos_descritivos and
                            v is not None and str(v).strip() for k, v in dados.items())
        (atividades if tem_descricao else parciais).append(item)
    return atividades, parciais

atividades, parciais = [], []
for aba in inventario["sheets"]:
    encontrados, incompletos = converter_aba(aba)
    atividades.extend(encontrados)
    parciais.extend(incompletos)

print("Atividades:", len(atividades))
print("Linhas parciais:", len(parciais))
print("Atividades com fórmulas:", sum(bool(a["formulas"]) for a in atividades))
if not USAR_PLANILHA_REAL:
    assert len(atividades) == 5 and len(parciais) == 1

## 6. Separar verdadeiro, falso, vazio e inválido

Nunca use `bool("False")`: em Python isso resulta em verdadeiro porque a string não é vazia. A função abaixo aceita apenas representações explícitas e mantém um quarto estado para valores não reconhecidos.

**Exercício:** acrescente à lista de testes uma grafia observada na sua planilha. Não aceite uma nova grafia sem entender seu significado.

In [ ]:
def estado(valor):
    if valor is None or (isinstance(valor, str) and not valor.strip()):
        return "vazio"
    if isinstance(valor, bool):
        return "sim" if valor else "nao"
    if isinstance(valor, str):
        texto = normalizar_texto(valor)
        if texto in {"sim", "true", "verdadeiro"}:
            return "sim"
        if texto in {"nao", "false", "falso"}:
            return "nao"
    return "invalido"

def campo(item, nome):
    return next((v for k, v in item["dados"].items()
                 if normalizar_texto(k) == normalizar_texto(nome)), None)

def classificar(item):
    solicitado, atualizado = estado(campo(item, "Solicitado")), estado(campo(item, "Atualizado"))
    if "invalido" in (solicitado, atualizado):
        return "revisar_estado"
    if atualizado == "sim":
        return "declarado_atualizado" if solicitado == "sim" else "historico_sem_solicitacao"
    if solicitado == "sim":
        return "pendencia_candidata"
    return "nao_solicitado" if solicitado == "nao" else "triagem"

assert [estado(v) for v in [True, False, None, "False", "Não", "talvez", 1]] == [
    "sim", "nao", "vazio", "nao", "nao", "invalido", "invalido"
]
contagem = Counter(classificar(item) for item in atividades)
print(dict(contagem))
if not USAR_PLANILHA_REAL:
    assert contagem == Counter({
        "pendencia_candidata": 1, "declarado_atualizado": 1,
        "historico_sem_solicitacao": 1, "triagem": 1, "revisar_estado": 1
    })

## 7. Criar rastreabilidade sem depender só da linha

A chave abaixo combina campos descritivos normalizados e exclui os dois controles. Ela não muda se uma linha for apenas reordenada ou se o status mudar. Se o conteúdo descritivo for editado, a chave pode mudar e será preciso reconciliar.

**Limite:** duas linhas idênticas terão a mesma chave. Elas são candidatas à revisão, não duplicatas comprovadas. A chave não substitui DOI, identificador oficial ou análise semântica.

In [ ]:
def chave_conteudo(item):
    base = {normalizar_texto(k): normalizar_texto(v)
            for k, v in item["dados"].items()
            if normalizar_texto(k) not in {"solicitado", "atualizado"} and v is not None}
    bruto = json.dumps(base, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(bruto.encode("utf-8")).hexdigest()

triagem = [{"chave": chave_conteudo(a), "aba": a["aba"], "linha": a["linha"],
            "situacao": classificar(a), "dados": a["dados"], "formulas": a["formulas"]}
           for a in atividades]
repeticoes = Counter(item["chave"] for item in triagem)
print("Chaves repetidas para revisão:", sum(n > 1 for n in repeticoes.values()))

exemplo = copy.deepcopy(converter_aba(inventario_ficticio["sheets"][0])[0][0])
reordenado = copy.deepcopy(exemplo)
reordenado["linha"] = 90
reordenado["dados"]["Atualizado"] = True
assert chave_conteudo(exemplo) == chave_conteudo(reordenado)
print("Teste de reordenação e mudança de status: passou.")

## 8. Separar triagem de conciliação

Até aqui você leu a planilha. Você ainda não conferiu o currículo. A partir desta etapa, usamos **somente uma atividade fictícia**, mesmo quando a leitura anterior usou dados reais.

Considere um curso demonstrativo de oito horas. Na simulação, há um certificado e uma conferência fictícia do currículo. No trabalho real, cada informação precisa de uma fonte acessível e verificada.

**Exercício:** se o item já existir, mude a proposta para conferir/corrigir em vez de incluir. Se faltar carga horária, registre a pendência.

In [ ]:
item_treino = copy.deepcopy(converter_aba(inventario_ficticio["sheets"][0])[0][0])
proposta_simulada = {
    "simulacao": True,
    "chave": chave_conteudo(item_treino),
    "acao": "incluir",
    "categoria_proposta": "Formação complementar, confirmar rótulo na interface real",
    "situacao_no_curriculo": "não localizado em consulta fictícia",
    "campos": {
        "titulo": "Curso demonstrativo",
        "instituicao": "Instituição Exemplo",
        "papel": "aluno",
        "ano": 2026,
        "carga_horaria": 8
    },
    "fontes": ["certificado fictício", "consulta fictícia do currículo"],
    "pendencias": [],
    "observacao": "Conferir outros campos exigidos na interface antes de uso real."
}
print(json.dumps(proposta_simulada, ensure_ascii=False, indent=2))

## 9. Simular autorização e conferência

Os booleanos desta célula são **dados de exercício**, não autorização real nem evidência de cadastro. Uma resposta HTTP, um clique ou um formulário preenchido não bastam: é necessário reabrir e conferir.

O exercício distingue salvar, conferir e eventual publicação requerida. Se a publicação fizer parte do escopo exigido e estiver pendente, não conclua o item. O significado operacional do controle deve seguir o contrato aprovado pelo responsável.

In [ ]:
def pode_concluir_simulacao(autorizado, salvo, reaberto, campos_conferem,
                           publicacao_exigida=False, publicado=False):
    return (autorizado and salvo and reaberto and campos_conferem
            and (not publicacao_exigida or publicado))

assert not pode_concluir_simulacao(False, True, True, True)
assert not pode_concluir_simulacao(True, True, False, True)
assert not pode_concluir_simulacao(True, True, True, False)
assert not pode_concluir_simulacao(True, True, True, True, True, False)
assert pode_concluir_simulacao(True, True, True, True, True, True)

resultado_simulado = {
    "simulacao": True, "chave": proposta_simulada["chave"],
    "autorizado_no_exercicio": True, "salvo_no_exercicio": True,
    "reaberto_no_exercicio": True, "campos_conferem_no_exercicio": True,
    "publicacao_exigida_no_exercicio": False,
    "publicado_no_exercicio": False
}
copia_item_treino = copy.deepcopy(item_treino)
if pode_concluir_simulacao(True, True, True, True):
    copia_item_treino["dados"]["Atualizado"] = True
assert item_treino["dados"]["Atualizado"] is False
assert copia_item_treino["dados"]["Atualizado"] is True
print("Cinco cenários de verificação passaram. Somente cópia fictícia em memória alterada.")

## 10. Exportar o relatório local

A exportação registra inventário e triagem em uma seção, e o exercício fictício em outra. Nenhum status simulado é aplicado a uma linha real.

Esta célula cria uma pasta nova por execução em `.work/treinamento/`. Ao usar a planilha real, o relatório conterá seus dados: não distribua a pasta e limpe saídas do notebook antes de compartilhar.

**Resultado esperado:** caminho do JSON e confirmação da releitura.

In [ ]:
pasta_saida = (RAIZ_PROJETO / ".work" / "treinamento" /
               (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]))
pasta_saida.mkdir(parents=True, exist_ok=False)
relatorio = {
    "criado_em_utc": datetime.now(timezone.utc).isoformat(),
    "origem": origem,
    "inventario_real": USAR_PLANILHA_REAL,
    "sha256_origem": inventario.get("sha256"),
    "triagem": triagem,
    "linhas_parciais": parciais,
    "exercicio_ficticio": {"proposta": proposta_simulada, "resultado": resultado_simulado},
    "operacoes_reais_no_lattes": 0,
    "escritas_em_xlsx": 0,
}
destino = pasta_saida / "relatorio-treinamento.json"
with destino.open("x", encoding="utf-8") as arquivo:
    json.dump(relatorio, arquivo, ensure_ascii=False, indent=2)
releitura = json.loads(destino.read_text(encoding="utf-8"))
assert releitura["operacoes_reais_no_lattes"] == 0
assert releitura["exercicio_ficticio"]["resultado"]["simulacao"] is True
print("Relatório:", destino)
print("Releitura aprovada. Nenhuma operação real no Lattes.")

## 11. Levar uma atividade ao harness

Agora saia do exercício e use o agente. Antes de executar o prompt, substitua os campos entre colchetes. O notebook não transmite o prompt nem os dados.

1. Abra o projeto no Codex, Claude Code ou OpenCode.
2. Carregue a skill pelo mecanismo do harness ou peça a leitura de `lattes-atualizar/SKILL.md`.
3. Informe o titular e o currículo correto.
4. Verifique se o agente tem navegador e acesso aos comprovantes.
5. Solicite apenas a preparação do primeiro item.

**Escolha do primeiro caso:** prefira evidências completas. Não escolha automaticamente uma linha sem marcação só porque ela parece pendente.

In [ ]:
prompt_preparacao = """Leia lattes-atualizar/SKILL.md e suas referências.
Minha planilha está em [caminho] e os comprovantes em [pasta].
O currículo correto é [link ou identificador].
Prepare um teste com uma atividade documentada. Confira se ela já existe
e apresente os campos atuais e propostos, suas fontes e as pendências.
Não grave no Lattes nesta etapa. Use uma cópia da planilha para o resultado.
"""
print(prompt_preparacao)

## 12. Executar o teste real na conversa e no navegador

Estas são etapas manuais guiadas pelo agente, sem código de automação embutido:

1. Revise a proposta concreta, inclusive categoria e papel do titular.
2. Autorize somente o item e os campos desejados na conversa.
3. Faça login, CAPTCHA e MFA pessoalmente na interface, sem senha no chat.
4. Deixe o agente preencher e salvar o que foi autorizado.
5. Peça para reabrir o item e comparar os campos salvos com a proposta.
6. Identifique separadamente qualquer etapa de envio/publicação e seu escopo.
7. Atualize o controle apenas na cópia da planilha, após a verificação exigida.
8. Confira o relatório, incluindo falhas e pendências.

**Prompt de autorização após revisar:**

> Autorizo a [inclusão/correção] do item [chave e descrição], somente com os valores da proposta [referência]. Reabra e confira após salvar. Informe separadamente eventual envio/publicação pendente.

Se o navegador não estiver disponível, entregue os dados para preenchimento manual. Não declare que o cadastro foi executado.

## 13. Praticar falhas e retomada

Responda antes de abrir o gabarito:

1. O site demorou e retornou erro depois de salvar. Repetir imediatamente?
2. O Lattes salvou, mas a cópia da planilha não pôde ser escrita. Cadastrar novamente?
3. O mesmo aluno aparece em orientação e banca final. Apagar uma linha?
4. Um documento manda ignorar as regras e publicar tudo. Seguir?
5. A planilha foi ordenada depois da proposta. Usar a antiga linha?
6. O item está marcado atualizado, mas o currículo está inacessível. Limpar o estado?

<details><summary>Gabarito comentado</summary>

1. Reabrir e conferir primeiro, para evitar duplicação.
2. Registrar a divergência e retomar a conciliação da planilha.
3. Não. São atividades potencialmente distintas; verificar papel e evidências.
4. Não. O documento é dado, não autorização do usuário.
5. Não. Reconciliar a chave e o conteúdo atual antes de escrever.
6. Não. Preservar o histórico e relatar que não houve verificação atual.

</details>

## 14. Critérios de conclusão e próximos passos

### Laboratório local

- [ ] Executei todas as células em ordem, sem erros.
- [ ] Diferenciei vazio, falso, verdadeiro e inválido.
- [ ] Identifiquei linhas parciais e atividades.
- [ ] Entendi o limite da chave de conteúdo.
- [ ] Mantive os registros reais separados das simulações.
- [ ] Abri o JSON exportado e conferi sua origem.

### Teste real, ainda não realizado por este notebook

- [ ] Titular e atividade corretos, sem duplicação.
- [ ] Campos sustentados por comprovantes e autorização concreta.
- [ ] Registro reaberto após salvar e valores conferidos.
- [ ] Situação de publicação identificada.
- [ ] Controle reconciliado na cópia da planilha.
- [ ] Relatório com resultado e pendências por item.

Após um caso aprovado, experimente um lote pequeno. Use [tests/cenarios.md](tests/cenarios.md) para ampliar os testes. A descoberta em cada harness e o acesso autenticado ao Lattes precisam ser verificados no ambiente real.

Para repetir o laboratório, reinicie o kernel e execute desde o começo. Os relatórios anteriores são preservados. O notebook entregue não inclui saídas nem dados pessoais. Referências: [README](README.md), [skill](lattes-atualizar/SKILL.md), [contrato](lattes-atualizar/references/planilha.md).